# 02 — Data Cleaning & Quality Assurance

**Input:** `data/raw/PowerGrid_Utility_Intelligence.csv`  
**Output:** `data/processed/clean.csv`

### Steps covered
1. Load raw data and confirm shape
2. Normalize `asset_type` casing
3. Remove duplicate rows
4. Impute missing values (grouped median by asset_type)
5. Winsorize outliers (1st–99th percentile)
6. Drop administrative/ID columns
7. Final quality check and export

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
print('Libraries loaded.')

Libraries loaded.


## Step 1 — Load Raw Data

In [2]:
df = pd.read_csv('../data/raw/PowerGrid_Utility_Intelligence.csv')
print(f'Shape: {df.shape}')   # expect (50500, 32)
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Shape: (50500, 32)
Columns: ['asset_id', 'asset_type', 'asset_age_years', 'substation_region', 'manufacturer', 'power_load_mw', 'load_utilization_pct', 'voltage_fluctuation_pct', 'frequency_deviation_hz', 'transformer_temperature_c', 'oil_quality_score', 'equipment_health_score', 'vibration_level', 'maintenance_overdue_days', 'maintenance_events_last_12m', 'inspection_score', 'previous_outages_12m', 'avg_outage_duration_minutes', 'customer_complaints_last_12m', 'temperature_c', 'humidity_pct', 'wind_speed_kmh', 'rainfall_mm', 'storm_risk_index', 'customers_served', 'estimated_revenue_loss', 'regulatory_penalty_cost', 'legacy_asset_code', 'grid_cluster_id', 'monitoring_batch_id', 'administrative_reference', 'grid_failure_flag']


,asset_id,asset_type,asset_age_years,substation_region,manufacturer,power_load_mw,load_utilization_pct,voltage_fluctuation_pct,frequency_deviation_hz,transformer_temperature_c,oil_quality_score,equipment_health_score,vibration_level,maintenance_overdue_days,maintenance_events_last_12m,inspection_score,previous_outages_12m,avg_outage_duration_minutes,customer_complaints_last_12m,temperature_c,humidity_pct,wind_speed_kmh,rainfall_mm,storm_risk_index,customers_served,estimated_revenue_loss,regulatory_penalty_cost,legacy_asset_code,grid_cluster_id,monitoring_batch_id,administrative_reference,grid_failure_flag
0,100000,Transmission Line,39,Central,ABB,282.834,81.260,1.411,-0.370,66.341,23.993,69.502,0.005,65,3,48.518,4,38.221,8,20.882,47.889,13.201,19.457,66.896,39815,586776.071,340322.797,139462,36,1711,810874,0
1,100001,Switchgear,29,Central,Schneider,113.757,70.554,7.717,0.375,76.477,50.628,78.042,18.395,100,5,84.032,3,92.172,5,48.103,65.405,103.514,13.165,39.487,47370,801405.346,279753.232,336586,37,5380,257421,0
2,100002,Transmission Line,15,North,ABB,77.586,39.119,11.181,0.024,68.775,NaN,19.769,9.397,104,5,52.108,2,36.029,6,49.332,25.241,90.239,32.955,58.143,25676,251440.743,355793.399,964465,36,7231,428171,1


## Step 2 — Normalize `asset_type` Casing

EDA found 3 casing variants of 'Transformer': `'Transformer'`, `'TRANSFORMER'`, `'transformer'`.  
`.str.title()` converts all to title case consistently.

In [3]:
print('Before normalization:')
print(df['asset_type'].value_counts())

df['asset_type'] = df['asset_type'].str.strip().str.title()

print('\nAfter normalization:')
print(df['asset_type'].value_counts())
assert df['asset_type'].nunique() == 5, f'Expected 5 asset types, got {df["asset_type"].nunique()}'

Before normalization:
asset_type
Transformer          10253
Switchgear            9805
Circuit Breaker       9779
Substation            9662
Transmission Line     9646
TRANSFORMER            715
transformer            640
Name: count, dtype: int64

After normalization:
asset_type
Transformer          11608
Switchgear            9805
Circuit Breaker       9779
Substation            9662
Transmission Line     9646
Name: count, dtype: int64


## Step 3 — Remove Duplicate Rows

EDA confirmed exactly 500 duplicate rows (~1%). Keeping first occurrence.

In [4]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
removed = before - after
print(f'Removed {removed} duplicate rows.')
print(f'New shape: {df.shape}')
assert removed == 500, f'Expected 500 duplicates, removed {removed}'

Removed 500 duplicate rows.
New shape: (50000, 32)


## Step 4 — Handle Missing Values

Two columns have ~10% missing values:
- `oil_quality_score`: not applicable for non-transformer assets
- `inspection_score`: inspections not yet completed for some assets

**Strategy:** Impute with median grouped by `asset_type`.  
Using grouped median (not global) because different asset types operate under very different conditions — a transformer's oil quality median is meaningless for a circuit breaker.

In [5]:
impute_cols = ['oil_quality_score', 'inspection_score']

print('Missing values before imputation:')
print(df[impute_cols].isnull().sum())

for col in impute_cols:
    df[col] = df.groupby('asset_type')[col].transform(
        lambda x: x.fillna(x.median())
    )

print('\nMissing values after imputation:')
print(df[impute_cols].isnull().sum())
assert df[impute_cols].isnull().sum().sum() == 0, 'Still have missing values after imputation'

Missing values before imputation:
oil_quality_score    4990
inspection_score     5047
dtype: int64



Missing values after imputation:
oil_quality_score    0
inspection_score     0
dtype: int64


## Step 5 — Treat Outliers (Winsorization)

EDA identified outliers in 3 columns via IQR method:
- `rainfall_mm` (~3.16%)
- `avg_outage_duration_minutes` (~2.53%)
- `power_load_mw` (~2.04%)

**Strategy:** Cap at 1st and 99th percentile (Winsorization).  
We do NOT delete rows because extreme weather events are real and likely caused failures. Winsorization preserves the record but limits disproportionate influence on model weights.

In [6]:
outlier_cols = ['rainfall_mm', 'avg_outage_duration_minutes', 'power_load_mw']

clip_bounds = {}
for col in outlier_cols:
    lo = df[col].quantile(0.01)
    hi = df[col].quantile(0.99)
    clip_bounds[col] = (lo, hi)
    before_min, before_max = df[col].min(), df[col].max()
    df[col] = df[col].clip(lower=lo, upper=hi)
    print(f'{col}: [{before_min:.2f}, {before_max:.2f}] → clipped to [{lo:.2f}, {hi:.2f}]')

print('\nWinsorization complete.')

rainfall_mm: [0.07, 211.34] → clipped to [2.19, 97.66]
avg_outage_duration_minutes: [1.39, 305.90] → clipped to [8.74, 168.12]
power_load_mw: [2.81, 418.04] → clipped to [25.53, 233.56]

Winsorization complete.


## Step 6 — Assess and Drop Administrative Columns

Four columns are system/administrative IDs with no operational meaning. We evaluate their correlation with `grid_failure_flag` and drop those with `|r| < 0.05`.  
We also drop `asset_id` — it's a unique row identifier, not a predictive feature.

In [7]:
admin_cols = ['legacy_asset_code', 'grid_cluster_id',
              'monitoring_batch_id', 'administrative_reference']

corr_with_target = df[admin_cols + ['grid_failure_flag']].corr()['grid_failure_flag'].drop('grid_failure_flag')
print('Correlation of admin columns with grid_failure_flag:')
print(corr_with_target.round(4))

to_drop = [col for col in admin_cols if abs(corr_with_target[col]) < 0.05]
to_drop.append('asset_id')  # unique identifier — not a feature

df.drop(columns=to_drop, inplace=True, errors='ignore')
print(f'\nDropped columns: {to_drop}')
print(f'Remaining shape: {df.shape}')

Correlation of admin columns with grid_failure_flag:
legacy_asset_code          -0.003
grid_cluster_id             0.003
monitoring_batch_id         0.003
administrative_reference   -0.000
Name: grid_failure_flag, dtype: float64

Dropped columns: ['legacy_asset_code', 'grid_cluster_id', 'monitoring_batch_id', 'administrative_reference', 'asset_id']
Remaining shape: (50000, 27)


## Step 7 — Final Quality Check

In [8]:
print('=== Final Quality Report ===')
print(f'Shape:             {df.shape}')
print(f'Duplicate rows:    {df.duplicated().sum()}')
missing = df.isnull().sum()
print(f'Missing values:    {missing[missing > 0].to_dict() or "None"}')
print(f'asset_type values: {sorted(df["asset_type"].unique())}')
print(f'Target distribution:')
print(df['grid_failure_flag'].value_counts(normalize=True).round(3))

=== Final Quality Report ===
Shape:             (50000, 27)
Duplicate rows:    0
Missing values:    None
asset_type values: ['Circuit Breaker', 'Substation', 'Switchgear', 'Transformer', 'Transmission Line']
Target distribution:
grid_failure_flag
0   0.570
1   0.430
Name: proportion, dtype: float64


In [9]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/clean.csv', index=False)
print(f'Saved: data/processed/clean.csv  |  Shape: {df.shape}')

Saved: data/processed/clean.csv  |  Shape: (50000, 27)
